# 06_v5c_3_1_comparison — V5C 3.0 vs V5C 3.1 对比回测

> 目的：评估"去单股(GOOG+AMZN)，全部并入 QQQ"对组合表现的影响

**结构（修正后分类）**：
- 进攻 50%: VOO / QQQ / HQH / XLV + 单股 (GOOG/AMZN)
- 对冲 30%: GLDM / BCX (商品 CEF)
- 防御 20%: VGSH

**V5C 3.0 → V5C 3.1 变化**：
- 进攻类内部：GOOG 5% + AMZN 5% → QQQ 10%
- 对冲类、防御类不变
- **大类比例 50/30/20 维持不变**

**期望发现**：
- CAGR 略低（指数化降低 idiosyncratic alpha）
- Vol 略降（QQQ 比单股更分散）
- Sharpe 持平或略升（vol 降幅 > CAGR 降幅则 Sharpe 升）
- Max DD 应改善（去除单股归零风险）

**回测窗口**: 2011-01-01 至今 (~14.6 年)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 长周期代理映射
tickers_long = {
    'VOO': 'VFINX',
    'QQQ': 'QQQ',
    'HQH': 'HQH',
    'XLV': 'XLV',
    'BCX': 'BCX',
    'GLDM': 'GLD',
    'VGSH': 'VFITX',
    'GOOG': 'GOOG',
    'AMZN': 'AMZN',
}

start_date = '2011-01-01'
raw_data = yf.download(list(tickers_long.values()), start=start_date, auto_adjust=True)['Close']
rename_map = {v: k for k, v in tickers_long.items()}
raw_data.columns = [rename_map.get(c, c) for c in raw_data.columns]
data = raw_data.dropna()
returns = data.pct_change().dropna()
print(f'数据范围: {data.index[0].date()} -> {data.index[-1].date()}')
print(f'交易日数: {len(data)} ({len(data)/252:.1f} 年)')
print(f'\n标的: {list(data.columns)}')

In [ ]:
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used_tickers = [t for t in target_weights.keys() if t in returns_df.columns]
    sub_returns = returns_df[used_tickers]
    target = np.array([target_weights[t] for t in used_tickers])
    current_weights = target.copy()
    portfolio_returns = []
    rebalance_dates = [sub_returns.index[0]]
    
    for date, daily_ret in sub_returns.iterrows():
        port_ret = np.sum(current_weights * daily_ret.values)
        portfolio_returns.append(port_ret)
        new_weights = current_weights * (1 + daily_ret.values)
        new_weights = new_weights / new_weights.sum()
        max_dev_pp = np.max(np.abs(new_weights - target)) * 100
        if max_dev_pp >= threshold_pp:
            current_weights = target.copy()
            rebalance_dates.append(date)
        else:
            current_weights = new_weights
    
    return pd.Series(portfolio_returns, index=sub_returns.index), rebalance_dates

def compute_metrics(returns_series, rebalance_dates, name='Portfolio'):
    cum = (1 + returns_series).cumprod()
    n_years = len(returns_series) / 252
    cagr = cum.iloc[-1] ** (1/n_years) - 1
    vol = returns_series.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol
    downside = returns_series[returns_series < 0]
    sortino = (cagr - 0.04) / (downside.std() * np.sqrt(252))
    rolling_max = cum.expanding().max()
    drawdown = (cum / rolling_max) - 1
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd)
    return {
        'Name': name, 'CAGR': cagr, 'Vol': vol,
        'Sharpe': sharpe, 'Sortino': sortino,
        'Max DD': max_dd, 'Calmar': calmar,
        'Rebalances': len(rebalance_dates) - 1,
        'Years': n_years,
    }

In [ ]:
# ============================================================
# 定义两个组合 (注意分类: 单股属于进攻, BCX属于对冲)
# ============================================================

# V5C 3.0 (修正分类)
# 进攻 50%: VOO 15 / QQQ 5 / HQH 10 / XLV 10 / GOOG 5 / AMZN 5
# 对冲 30%: GLDM 20 / BCX 10
# 防御 20%: VGSH 20
V5C_3_0 = {
    'VOO': 0.15, 'QQQ': 0.05, 'HQH': 0.10, 'XLV': 0.10,
    'GOOG': 0.05, 'AMZN': 0.05,
    'GLDM': 0.20, 'BCX': 0.10,
    'VGSH': 0.20,
}

# V5C 3.1: 单股全部并入 QQQ, 其他不变
# 进攻 50%: VOO 15 / QQQ 15 / HQH 10 / XLV 10
# 对冲 30%: GLDM 20 / BCX 10 (不变)
# 防御 20%: VGSH 20 (不变)
V5C_3_1 = {
    'VOO': 0.15, 'QQQ': 0.15, 'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.20, 'BCX': 0.10,
    'VGSH': 0.20,
}

print(f'V5C 3.0 权重和: {sum(V5C_3_0.values()):.2f}')
print(f'V5C 3.1 权重和: {sum(V5C_3_1.values()):.2f}')

# 验证分类
print('\n分类验证 V5C 3.0:')
offense_30 = V5C_3_0['VOO'] + V5C_3_0['QQQ'] + V5C_3_0['HQH'] + V5C_3_0['XLV'] + V5C_3_0['GOOG'] + V5C_3_0['AMZN']
hedge_30 = V5C_3_0['GLDM'] + V5C_3_0['BCX']
defense_30 = V5C_3_0['VGSH']
print(f'  进攻 {offense_30:.0%} / 对冲 {hedge_30:.0%} / 防御 {defense_30:.0%}')

print('\n分类验证 V5C 3.1:')
offense_31 = V5C_3_1['VOO'] + V5C_3_1['QQQ'] + V5C_3_1['HQH'] + V5C_3_1['XLV']
hedge_31 = V5C_3_1['GLDM'] + V5C_3_1['BCX']
defense_31 = V5C_3_1['VGSH']
print(f'  进攻 {offense_31:.0%} / 对冲 {hedge_31:.0%} / 防御 {defense_31:.0%}')

In [ ]:
# ============================================================
# 跑回测
# ============================================================
ret_30, dates_30 = simulate_rebalance(returns, V5C_3_0, threshold_pp=5.0)
ret_31, dates_31 = simulate_rebalance(returns, V5C_3_1, threshold_pp=5.0)

m30 = compute_metrics(ret_30, dates_30, 'V5C 3.0 (含单股)')
m31 = compute_metrics(ret_31, dates_31, 'V5C 3.1 (无单股)')

print('=' * 78)
print('指标对比 (±5pp 阈值再平衡, 14.6 年回测)')
print('=' * 78)
for col in ['CAGR', 'Vol', 'Sharpe', 'Sortino', 'Max DD', 'Calmar']:
    fmt = '{:.2%}' if col not in ['Sharpe', 'Sortino', 'Calmar'] else '{:.3f}'
    v30 = m30[col]
    v31 = m31[col]
    delta = v31 - v30
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'  {col:<10}  V5C 3.0: {fmt.format(v30):>10}  V5C 3.1: {fmt.format(v31):>10}  Δ: {fmt.format(delta):>10}  {arrow}')

print(f'\n  Rebalances: V5C 3.0: {m30["Rebalances"]}  V5C 3.1: {m31["Rebalances"]}')

In [ ]:
# ============================================================
# 净值曲线 + 回撤对比
# ============================================================
cum_30 = (1 + ret_30).cumprod()
cum_31 = (1 + ret_31).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(cum_30.index, cum_30.values, label='V5C 3.0 (含 GOOG+AMZN)', linewidth=2, alpha=0.85)
axes[0].plot(cum_31.index, cum_31.values, label='V5C 3.1 (并入 QQQ)', linewidth=2, alpha=0.85)
axes[0].set_title('净值曲线对比 (起始 = 1.0, log scale)', fontsize=14)
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_yscale('log')

rm30 = cum_30.expanding().max()
dd30 = (cum_30 / rm30) - 1
rm31 = cum_31.expanding().max()
dd31 = (cum_31 / rm31) - 1

axes[1].fill_between(dd30.index, dd30.values, 0, alpha=0.4, label='V5C 3.0')
axes[1].fill_between(dd31.index, dd31.values, 0, alpha=0.4, label='V5C 3.1')
axes[1].set_title('回撤对比', fontsize=14)
axes[1].set_ylabel('Drawdown')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 危机期表现
# ============================================================
crises = {
    '2018-Q4 (10-12月)':    ('2018-10-01', '2018-12-31'),
    '2020 COVID (2-4月)':   ('2020-02-19', '2020-04-30'),
    '2022 Bear (1-10月)':   ('2022-01-01', '2022-10-31'),
    '2025 Q1 (Tariff/Tech)':('2025-01-01', '2025-04-30'),
}

print('=' * 78)
print('危机期间表现对比')
print('=' * 78)
for name, (s, e) in crises.items():
    c30 = (1 + ret_30.loc[s:e]).prod() - 1
    c31 = (1 + ret_31.loc[s:e]).prod() - 1
    delta = c31 - c30
    print(f'\n{name}:')
    print(f'  V5C 3.0: {c30:+7.2%}')
    print(f'  V5C 3.1: {c31:+7.2%}')
    print(f'  Δ:       {delta:+7.2%}  {"V5C 3.1 表现更好" if delta > 0 else "V5C 3.1 表现更差"}')

## 解读模板（运行后填空）

1. **CAGR 变化**：____ → ____
2. **Sharpe 变化**：____ → ____
3. **Max DD 变化**：____ → ____
4. **Calmar 变化**：____ → ____
5. **再平衡次数**：____ → ____

## 决策标准

由于大类比例不变，差异完全来自"指数 vs 单股"：

- **若 Sharpe 改善**：直接采纳 V5C 3.1（单股拖累已得到改善）
- **若 Sharpe 持平、Max DD 改善**：采纳（单股归零风险消除是隐性价值）
- **若 Sharpe 略降但 Max DD 改善**：采纳（用 alpha 换确定性合理）
- **若 Sharpe 显著降 + Max DD 也变差**：重新考虑（说明单股历史上是 alpha 来源）